# 03. Kompleksowy Feature Engineering

Poniższy notatnik wykonuje inżynierię cech (dla sieci LSTM / Transformers) pomijając celowy target `x2` oraz dynamicznie interpoluje w czasie i przestrzeni zewnętrzne dane pogodowe z pliku `weather.csv`.


In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
import warnings

warnings.filterwarnings('ignore')

## 1. Wczytanie Baz

In [2]:
TRAIN_PATH = 'data/data_train.csv'
DEVICES_PATH = 'data/devices.csv'
WEATHER_PATH = 'data/weather.csv'

train_lz = pl.scan_csv(TRAIN_PATH)
devices_lz = pl.scan_csv(DEVICES_PATH)
weather_lz = pl.scan_csv(WEATHER_PATH)

## 2. Dopasowanie wielowymiarowe Danych Pogodowych (`ASOF-Join`)
Ponieważ odczyty sensorów `x1`, `t1` napływają co 5 minut, a pobrana pogoda z Open-Meteo to punktowe pomiary cogodzinne (`1H`), używamy wbudowanej łączącej funkcji stochastycznie-czasowej (`join_asof`). Operacja ta bierze pojedyncze auto o danym `deviceId` i przykleja mu zawsze zstępnie (backward) najbliższą dotychczas zanotowaną metrykę pogodową do owych 5-minutowych pomiarów.

In [3]:
# Parse czasu do obiektów Datetime niezbędnych pod łączenia:
train_lz = train_lz.with_columns([
    pl.col('timedate').str.to_datetime().alias('dt_obj')
]).sort(['deviceId', 'dt_obj'])

weather_lz = weather_lz.with_columns([
    pl.col('timedate').str.to_datetime().alias('dt_obj')
]).sort(['deviceId', 'dt_obj']).drop('timedate')

# Złączenie typu asof dla zrównania częstotliwości 1H w dół do 5m per stacja!
train_lz = train_lz.join_asof(
    weather_lz,
    on='dt_obj',
    by='deviceId',
    strategy='backward'  # Ładuje "najświeższą przeszłą" stację do dziur między 1H pomiarami.
)

## 3. Cechy Czasu (Cyclical Features)

In [4]:
train_lz = train_lz.with_columns([
    pl.col('dt_obj').dt.hour().alias('hour'),
    pl.col('dt_obj').dt.weekday().alias('day_of_week')
]).with_columns([
    (np.sin(2 * np.pi * pl.col('hour') / 24)).alias('hour_sin'),
    (np.cos(2 * np.pi * pl.col('hour') / 24)).alias('hour_cos'),
    (np.sin(2 * np.pi * pl.col('day_of_week') / 7)).alias('day_sin'),
    (np.cos(2 * np.pi * pl.col('day_of_week') / 7)).alias('day_cos'),
    (pl.col('day_of_week') >= 6).cast(pl.Int8).alias('is_weekend')
])

## 4. Transformacja Złotej Zasady (Stacjonarność i Log Returns)
Eliminujemy trend by sieć lepiej reagowała na dane.

In [5]:
EPS = 1e-6

train_lz = train_lz.with_columns([
    pl.col('t1').diff().over('deviceId').alias('t1_diff'),
    ((pl.col('x1') + EPS).log() - (pl.col('x1').shift(1) + EPS).log()).over('deviceId').alias('x1_log_return')
])

## 5. Okna Wsteczne (Lags Windows)

In [6]:
lags_to_produce = [1, 2, 3, 6, 12, 24]

train_lz = train_lz.with_columns([
    pl.col('x1').shift(l).over('deviceId').alias(f'x1_lag_{l}') for l in lags_to_produce
]).with_columns([
    pl.col('t1').shift(l).over('deviceId').alias(f't1_lag_{l}') for l in [1, 12]
])

## 6. GARCH clustering & Low-pass Filtry

In [7]:
train_lz = train_lz.with_columns([
    pl.col('x1_log_return').rolling_std(window_size=12).over('deviceId').alias('volatility_x1_1h'),
    pl.col('x1_log_return').rolling_std(window_size=288).over('deviceId').alias('volatility_x1_24h'),
    pl.col('x1').rolling_mean(window_size=6).over('deviceId').alias('x1_smoothed_30m')
])

## 7. Cechy Globalne (Makro Sieć Czuć) i Anomalie

In [ ]:
global_stats_lz = train_lz.group_by('dt_obj').agg([
    pl.col('x1').mean().alias('global_net_x1_mean'),
    pl.col('x1').std().alias('global_net_x1_std')
])

train_lz = train_lz.with_columns([
    (pl.col('x1_log_return').abs() > (3 * pl.col('volatility_x1_24h').fill_null(float('inf')))).cast(pl.Int8).alias('x1_anomaly_flag')
])